In [1]:
import pandas as pd
import numpy as np
import torch
import transformers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoModel, BertTokenizerFast, DistilBertTokenizerFast, AutoTokenizer
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.utils.data import Dataset
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm
from nltk.tokenize import sent_tokenize, word_tokenize
import os
import nltk
from sklearn.feature_extraction.text import CountVectorizer
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Read Data

In [4]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
train_df = pd.read_csv('../data/combined_letters_degendered_with_topics_train.csv')
val_df = pd.read_csv('../data/combined_letters_degendered_with_topics_val.csv')
test_df = pd.read_csv('../data/combined_letters_degendered_with_topics_test.csv')

# Create Training and Test Sets

In [5]:
bert = AutoModel.from_pretrained("distilbert/distilbert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [6]:
class TextWithTopicsDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.text_data = df['full_text'].tolist()
        self.topic_features = df.filter(like='topic_').values
        self.labels = df['label'].values
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.text_data)

    def __getitem__(self, idx):
        text = self.text_data[idx]
        topics = torch.tensor(self.topic_features[idx], dtype=torch.float)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        tokens = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )

        return {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'topic_feats': topics,
            'label': label
        }

In [7]:
train_dataset = TextWithTopicsDataset(train_df, tokenizer)
val_dataset = TextWithTopicsDataset(val_df, tokenizer)
test_dataset = TextWithTopicsDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

In [8]:
# freeze weights, uncomment if you would like to unfreeze weights
for param in bert.parameters():
    param.requires_grad = False


# Create Model

In [9]:
class BERTWithTopics(nn.Module):
    def __init__(self, bert, topic_feat_dim, num_classes):
        super(BERTWithTopics, self).__init__()
        self.bert = bert
        self.dropout = nn.Dropout(0.1)
        self.relu = nn.ReLU()

        combined_dim = self.bert.config.hidden_size + topic_feat_dim  # 768 + number of topic features
        self.fc1 = nn.Linear(combined_dim, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input_ids, attention_mask, topic_feats):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_embed = bert_out.last_hidden_state[:, 0]  # [CLS] token

        x = torch.cat((cls_embed, topic_feats), dim=1)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return self.softmax(x)

In [10]:
model = BERTWithTopics(bert, 97, 2)
model = model.to(device)

In [11]:
batch_size = 16
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_df['label']), y=train_df['label'])
weights= torch.tensor(class_weights,dtype=torch.float)
weights = weights.to(device)
cross_entropy  = nn.NLLLoss(weight=weights)
epochs = 10

In [12]:
# function to train the model
def train():

  model.train()

  total_loss, total_accuracy = 0, 0

  # empty list to save model predictions
  total_preds=[]
  total_labels=[]

  # iterate over batches
  for step,batch in enumerate(tqdm(train_loader, desc="Training", leave=True)):

    # # progress update after every 50 batches.
    # if step % 50 == 0 and not step == 0:
    #   print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(train_dataloader)))

    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    topic_feats = batch['topic_feats'].to(device)
    labels = batch['label'].to(device)

    # clear previously calculated gradients
    model.zero_grad()

    # get model predictions for the current batch
    preds = model(input_ids, attention_mask, topic_feats)

    # compute the loss between actual and predicted values
    loss = cross_entropy(preds, labels)

    # add on to the total loss
    total_loss = total_loss + loss.item()

    # backward pass to calculate the gradients
    loss.backward()

    # clip the the gradients to 1.0. It helps in preventing the exploding gradient problem
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    # update parameters
    optimizer.step()

    # model predictions are stored on GPU. So, push it to CPU
    preds=preds.detach().cpu().numpy()

    # append the model predictions
    total_preds.append(preds)

    labels=labels.detach().cpu().numpy()

    # append labels
    total_labels.append(labels)

  # compute the training loss of the epoch
  avg_loss = total_loss / len(train_loader)

  # predictions are in the form of (no. of batches, size of batch, no. of classes).
  # reshape the predictions in form of (number of samples, no. of classes)
  total_preds  = np.concatenate(total_preds, axis=0)

  total_labels = np.concatenate(total_labels, axis=0)

  epoch_preds = np.argmax(total_preds, axis = 1)

  print('Training Classification Report: \n', classification_report(total_labels, epoch_preds))
  print('Training Confusion Matrix: \n', confusion_matrix(total_labels, epoch_preds))

  #returns the loss and predictions
  return avg_loss, total_preds

In [13]:
# function for evaluating the model
def evaluate():

  print("\nEvaluating...")

  # deactivate dropout layers
  model.eval()

  total_loss, total_accuracy = 0, 0

  # empty list to save the model predictions
  total_preds = []
  total_labels = []

  # iterate over batches
  for step,batch in enumerate(val_loader):

    # # Progress update every 50 batches.
    # if step % 50 == 0 and not step == 0:

    #   # Calculate elapsed time in minutes.
    #   elapsed = format_time(time.time() - t0)

    #   # Report progress.
    #   print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(val_dataloader)))

    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    topic_feats = batch['topic_feats'].to(device)
    labels = batch['label'].to(device)

    # deactivate autograd
    with torch.no_grad():

      # model predictions
      preds = model(input_ids, attention_mask, topic_feats)

      # compute the validation loss between actual and predicted values
      loss = cross_entropy(preds,labels)

      total_loss = total_loss + loss.item()

      preds = preds.detach().cpu().numpy()

      total_preds.append(preds)

      labels = labels.detach().cpu().numpy()

      total_labels.append(labels)

  # compute the validation loss of the epoch
  avg_loss = total_loss / len(val_loader)

  # reshape the predictions in form of (number of samples, no. of classes)
  total_preds  = np.concatenate(total_preds, axis=0)

  total_labels = np.concatenate(total_labels, axis=0)

  epoch_preds = np.argmax(total_preds, axis = 1)

  print('Validation Classification Report: \n', classification_report(total_labels, epoch_preds))
  print('Validation Confusion Matrix: \n', confusion_matrix(total_labels, epoch_preds))

  return avg_loss, total_preds

In [14]:
# set initial loss to infinite
best_valid_loss = float('inf')

# empty lists to store training and validation loss of each epoch
train_losses=[]
valid_losses=[]

#for each epoch
for epoch in tqdm(range(epochs), desc="Training"):

    print('\n Epoch {:} / {:}'.format(epoch + 1, epochs))

    #train model
    train_loss, _ = train()

    #evaluate model
    valid_loss, _ = evaluate()

    #save the best model
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        print('Model Saved!')
        torch.save(model, '../saved_models/saved_model.pt')

    # append training and validation loss
    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    print(f'\nTraining Loss: {train_loss:.3f}')
    print(f'Validation Loss: {valid_loss:.3f}')

Training:   0%|          | 0/10 [00:00<?, ?it/s]


 Epoch 1 / 10


Training:   0%|          | 0/404 [00:00<?, ?it/s]

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.32      0.45      0.38      2006
           1       0.70      0.58      0.64      4457

    accuracy                           0.54      6463
   macro avg       0.51      0.51      0.51      6463
weighted avg       0.58      0.54      0.55      6463

Training Confusion Matrix: 
 [[ 896 1110]
 [1866 2591]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.37      0.35      0.36       223
           1       0.72      0.74      0.73       496

    accuracy                           0.62       719
   macro avg       0.54      0.54      0.54       719
weighted avg       0.61      0.62      0.61       719

Validation Confusion Matrix: 
 [[ 78 145]
 [131 365]]
Model Saved!

Training Loss: 0.693
Validation Loss: 0.691

 Epoch 2 / 10


Training:   0%|          | 0/404 [00:00<?, ?it/s]

...

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.39      0.51      0.44      2006
           1       0.74      0.64      0.69      4457

    accuracy                           0.60      6463
   macro avg       0.56      0.57      0.56      6463
weighted avg       0.63      0.60      0.61      6463

Training Confusion Matrix: 
 [[1017  989]
 [1611 2846]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.38      0.39      0.39       223
           1       0.72      0.71      0.72       496

    accuracy                           0.61       719
   macro avg       0.55      0.55      0.55       719
weighted avg       0.62      0.61      0.61       719

Validation Confusion Matrix: 
 [[ 88 135]
 [144 352]]
Model Saved!

Training Loss: 0.674
Validation Loss: 0.685

 Epoch 9 / 10


Training:   0%|          | 0/404 [00:00<?, ?it/s]

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.39      0.55      0.45      2006
           1       0.75      0.61      0.67      4457

    accuracy                           0.59      6463
   macro avg       0.57      0.58      0.56      6463
weighted avg       0.64      0.59      0.60      6463

Training Confusion Matrix: 
 [[1096  910]
 [1740 2717]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.33      0.73      0.45       223
           1       0.73      0.33      0.45       496

    accuracy                           0.45       719
   macro avg       0.53      0.53      0.45       719
weighted avg       0.60      0.45      0.45       719

Validation Confusion Matrix: 
 [[162  61]
 [334 162]]

Training Loss: 0.672
Validation Loss: 0.689

 Epoch 10 / 10


Training:   0%|          | 0/404 [00:00<?, ?it/s]

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.38      0.57      0.46      2006
           1       0.75      0.59      0.66      4457

    accuracy                           0.58      6463
   macro avg       0.57      0.58      0.56      6463
weighted avg       0.64      0.58      0.60      6463

Training Confusion Matrix: 
 [[1134  872]
 [1818 2639]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.38      0.39      0.38       223
           1       0.72      0.72      0.72       496

    accuracy                           0.62       719
   macro avg       0.55      0.55      0.55       719
weighted avg       0.62      0.62      0.62       719

Validation Confusion Matrix: 
 [[ 86 137]
 [138 358]]
Model Saved!

Training Loss: 0.671
Validation Loss: 0.685


# Test Model

In [15]:
model = torch.load('../saved_models/saved_model.pt', weights_only=False)

In [16]:

model.eval()  # Set model to eval mode

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        topic_feats = batch['topic_feats'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids, attention_mask, topic_feats)

        # Get predicted class (as indices)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [17]:
  print('Test Classification Report: \n', classification_report(all_labels, all_preds))
  print('Test Confusion Matrix: \n', confusion_matrix(all_labels, all_preds))

Test Classification Report: 
               precision    recall  f1-score   support

           0       0.37      0.40      0.39       557
           1       0.72      0.69      0.71      1241

    accuracy                           0.60      1798
   macro avg       0.55      0.55      0.55      1798
weighted avg       0.61      0.60      0.61      1798

Test Confusion Matrix: 
 [[224 333]
 [379 862]]
